### ЗАДАЧА: Панель сверки выплат партнерам по паттерну `MVC`

Команда finance operations сверяет выплаты партнерам после закрытия расчетного периода.
Для каждого кейса нужно хранить:
- `case_id` — идентификатор кейса;
- `partner` — партнер;
- `period` — расчетный период;
- `expected_payout` — ожидаемая выплата;
- `actual_payout` — фактическая выплата;
- `orders_count` — число заказов за период;
- `delta_amount` — разница между фактической и ожидаемой выплатой;
- `delta_percent` — относительное отклонение в процентах;
- `status` — статус кейса;
- `analyst` — аналитик;
- `reserve_amount` — сумма резерва, примененная к кейсу;
- `decision` — итоговое решение.

### Формулы

При создании и после изменения сумм нужно правильно считать:
- `delta_amount = actual_payout - expected_payout`
- `delta_percent = abs(delta_amount) / expected_payout * 100`, если `expected_payout > 0`, иначе `0.0`
- оба значения должны быть округлены до 2 знаков.

Также нужна сводка:
- количество кейсов по статусам;
- `total_negative_delta` — сумма модулей кейсов, где `delta_amount < 0`;
- `total_positive_delta` — сумма кейсов, где `delta_amount > 0`;
- `total_reserved` — суммарно примененный резерв.

### Статусы кейса

- `new`
- `investigating`
- `reserve_applied`
- `ready_to_close`
- `closed`
- `escalated`

### Бизнес-правила

- нельзя создать кейс с уже существующим `case_id`;
- нельзя назначить аналитика несуществующему кейсу;
- финальные кейсы (`closed`, `escalated`) нельзя менять дальше;
- начать расследование можно только из `new` и только если назначен `analyst`;
- применить резерв можно только из `investigating`;
- резерв можно применять только если `delta_amount < 0`;
- `reserve_amount` должен быть больше 0 и не превышать `abs(delta_amount)`;
- при применении резерва `actual_payout` увеличивается на сумму резерва, после чего нужно пересчитать `delta_amount` и `delta_percent`;
- перевод в `ready_to_close` возможен только из `investigating` или `reserve_applied`;
- перевод в `ready_to_close` возможен только если `abs(delta_amount) <= 0.01`;
- закрыть кейс можно только из `ready_to_close`;
- эскалировать кейс можно только из `investigating`, `reserve_applied` или `ready_to_close`.

In [6]:
from dataclasses import dataclass


initial_cases = [
    ("PR-100", "best-gadgets", "2026-03", 12500.0, 12000.0, 310),
    ("PR-101", "home-decor-plus", "2026-03", 9800.0, 10050.0, 205),
]

actions = [
    ("show",),
    ("investigate", "PR-100"),
    ("assign", "PR-100", "Olga"),
    ("investigate", "PR-100"),
    ("reserve", "PR-100", 300.0),
    ("ready", "PR-100"),
    ("reserve", "PR-100", 200.0),
    ("ready", "PR-100"),
    ("close", "PR-100", "reserve_applied"),
    ("create", "PR-102", "sport-zone", "2026-03", 4300.0, 3900.0, 90),
    ("assign", "PR-102", "Max"),
    ("investigate", "PR-102"),
    ("escalate", "PR-102"),
    ("show",),
]


@dataclass
class PartnerPayoutCase:
    case_id: str
    partner: str
    period: str
    expected_payout: float
    actual_payout: float
    orders_count: int
    delta_amount: float
    delta_percent: float
    status: str = "new"
    analyst: str | None = None
    reserve_amount: float = 0.0
    decision: str | None = None


class PartnerPayoutModel:
    def __init__(self):
        self.cases = {}
        self.done_statuses = {'closed','escalated'}

    def _calculate_delta_amount(self, expected_payout: float, actual_payout: float) -> float:
        return round(actual_payout-expected_payout , 2)

    def _calculate_delta_percent(self, expected_payout: float, delta_amount: float) -> float:
        if expected_payout == 0:
            return 0.0
        
        return round((abs(delta_amount) / expected_payout) * 100, 2)

    def add_case(self, case_id: str, partner: str, period: str, expected_payout: float, actual_payout: float, orders_count: int) -> None:
        if case_id in self.cases:
            raise ValueError('already exists')
        delta_amount = self._calculate_delta_amount(expected_payout, actual_payout)
        delta_percent = self._calculate_delta_percent(expected_payout, delta_amount)
        self.cases[case_id] = PartnerPayoutCase(
            case_id,
            partner,
            period,
            expected_payout,
            actual_payout,
            orders_count,
            delta_amount,
            delta_percent,
        )

    def assign_analyst(self, case_id: str, analyst: str) -> None:
        if case_id not in self.cases:
            raise ValueError("Case not found")
        if self.cases[case_id].status in self.done_statuses:
            raise ValueError('already done')
        self.cases[case_id].analyst = analyst

    def start_investigation(self, case_id: str) -> None:
        if case_id not in self.cases:
            raise ValueError("Case not found")
        if self.cases[case_id].status in self.done_statuses:
            raise ValueError('already done')
        if self.cases[case_id].status != 'new':
            raise ValueError('not new')
        if self.cases[case_id].analyst is None:
            raise ValueError('no analyst')
        self.cases[case_id].status = "investigating"

    def apply_reserve(self, case_id: str, reserve_amount: float) -> None:
        if case_id not in self.cases:
            raise ValueError("Case not found")
        if self.cases[case_id].status in self.done_statuses:
            raise ValueError('already done')
        if self.cases[case_id].status != 'investigating':
            raise ValueError('doesn`t investigate')
        if self.cases[case_id].delta_amount >= 0:
            raise ValueError('wrong amount')
        if self.cases[case_id].reserve_amount > 0:
            raise ValueError('wrong reserve_amount')
        if self.cases[case_id].reserve_amount > abs(self.cases[case_id].delta_amount):
            raise ValueError ('reserve_amount more than abs(delta_amount)')

        case = self.cases[case_id]
        case.reserve_amount += reserve_amount
        case.actual_payout -= reserve_amount
        case.delta_amount = self._calculate_delta_amount(case.expected_payout, case.actual_payout)
        case.delta_percent = self._calculate_delta_percent(case.expected_payout, case.delta_amount)
        case.status = "reserve_applied"

    def mark_ready_to_close(self, case_id: str) -> None:
        if case_id not in self.cases:
            raise ValueError("Case not found")
        if self.cases[case_id].status in self.done_statuses:
            raise ValueError('already done')
        if self.cases[case_id].status not in {'investigating','reserve_applied'}:
            raise ValueError('not investigating or reserve_applied')
        if abs(self.cases[case_id].delta_amount) > 0.01:
            raise ValueError('abs(self.cases[case_id].delta_amount) > 0.01')
        
        self.cases[case_id].status = "ready_to_close"

    def close_case(self, case_id: str, decision: str) -> None:
        if case_id not in self.cases:
            raise ValueError("Case not found")
        if self.cases[case_id].status in self.done_statuses:
            raise ValueError('already done')
        
        if self.cases[case_id].status != 'ready_to_close':
            raise ValueError('not ready to close')

        case = self.cases[case_id]
        case.status = "closed"
        case.decision = decision

    def escalate_case(self, case_id: str) -> None:
        if case_id not in self.cases:
            raise ValueError("Case not found")
        if self.cases[case_id].status in self.done_statuses:
            raise ValueError('already done')
        if self.cases[case_id].status not in {'investigating','reserve_applied','ready_to_close'}:
            raise ValueError('not investigating,reserve_applied,ready_to_close')
        self.cases[case_id].status = "escalated"

    def list_cases(self) -> list[str]:
        rows = []
        for case in self.cases.values():
            rows.append(
                f"{case.case_id} | {case.partner} | {case.period} | {case.expected_payout} | {case.actual_payout} | "
                f"{case.orders_count} | {case.delta_amount} | {case.delta_percent} | {case.status} | {case.analyst} | {case.reserve_amount} | {case.decision}"
            )
        return rows

    def summary(self) -> dict[str, float | int]:
        result = {
            "total_cases": 0,
            "total_negative_delta": 0.0,
            "total_positive_delta": 0.0,
            "total_reserved": 0.0,
        }
        for case in self.cases.values():
            result[case.status] = result.get(case.status, 0) + 1
            result["total_cases"] += 1
            if case.delta_amount < 0:
                result["total_positive_delta"] += case.delta_amount
            elif case.delta_amount > 0:
                result["total_negative_delta"] += case.delta_amount
            result["total_reserved"] += case.actual_payout
        return result


class PartnerPayoutView:
    @staticmethod
    def render_cases(rows: list[str]) -> None:
        print("Partner payout cases:")
        for row in rows:
            print(row)

    @staticmethod
    def render_summary(summary: dict[str, float | int]) -> None:
        print("Summary:", summary)

    @staticmethod
    def render_success(message: str) -> None:
        print("SUCCESS:", message)

    @staticmethod
    def render_error(message: str) -> None:
        print("ERROR:", message)


class PartnerPayoutController:
    def __init__(self, model: PartnerPayoutModel, view: PartnerPayoutView):
        self.model = model
        self.view = view

    def create_case(self, case_id: str, partner: str, period: str, expected_payout: float, actual_payout: float, orders_count: int) -> None:
        try:
            self.model.add_case(case_id, partner, period, expected_payout, actual_payout, orders_count)
            self.view.render_success(f"Case {case_id} created")
        except ValueError as error:
            self.view.render_error(str(error))

    def assign_analyst(self, case_id: str, analyst: str) -> None:
        try:
            self.model.assign_analyst(case_id, analyst)
            self.view.render_success(f"Analyst assigned to {case_id}")
        except ValueError as error:
            self.view.render_error(str(error))

    def start_investigation(self, case_id: str) -> None:
        try:
            self.model.start_investigation(case_id)
            self.view.render_success(f"Case {case_id} moved to investigating")
        except ValueError as error:
            self.view.render_error(str(error))

    def apply_reserve(self, case_id: str, reserve_amount: float) -> None:
        try:
            self.model.apply_reserve(case_id, reserve_amount)
            self.view.render_success(f"Reserve applied to {case_id}")
        except ValueError as error:
            self.view.render_error(str(error))

    def mark_ready_to_close(self, case_id: str) -> None:
        try:
            self.model.mark_ready_to_close(case_id)
            self.view.render_success(f"Case {case_id} ready to close")
        except ValueError as error:
            self.view.render_error(str(error))

    def close_case(self, case_id: str, decision: str) -> None:
        try:
            self.model.close_case(case_id, decision)
            self.view.render_success(f"Case {case_id} closed")
        except ValueError as error:
            self.view.render_error(str(error))

    def escalate_case(self, case_id: str) -> None:
        try:
            self.model.escalate_case(case_id)
            self.view.render_success(f"Case {case_id} escalated")
        except ValueError as error:
            self.view.render_error(str(error))

    def show_cases(self) -> None:
        self.view.render_cases(self.model.list_cases())
        self.view.render_summary(self.model.summary())


model = PartnerPayoutModel()
view = PartnerPayoutView()
controller = PartnerPayoutController(model, view)

for case_id, partner, period, expected_payout, actual_payout, orders_count in initial_cases:
    model.add_case(case_id, partner, period, expected_payout, actual_payout, orders_count)

for action in actions:
    if action[0] == "show":
        controller.show_cases()
    elif action[0] == "create":
        _, case_id, partner, period, expected_payout, actual_payout, orders_count = action
        controller.create_case(case_id, partner, period, expected_payout, actual_payout, orders_count)
    elif action[0] == "assign":
        _, case_id, analyst = action
        controller.assign_analyst(case_id, analyst)
    elif action[0] == "investigate":
        _, case_id = action
        controller.start_investigation(case_id)
    elif action[0] == "reserve":
        _, case_id, reserve_amount = action
        controller.apply_reserve(case_id, reserve_amount)
    elif action[0] == "ready":
        _, case_id = action
        controller.mark_ready_to_close(case_id)
    elif action[0] == "close":
        _, case_id, decision = action
        controller.close_case(case_id, decision)
    elif action[0] == "escalate":
        _, case_id = action
        controller.escalate_case(case_id)

print()
print("Финальное состояние:")
controller.show_cases()


Partner payout cases:
PR-100 | best-gadgets | 2026-03 | 12500.0 | 12000.0 | 310 | -500.0 | 4.0 | new | None | 0.0 | None
PR-101 | home-decor-plus | 2026-03 | 9800.0 | 10050.0 | 205 | 250.0 | 2.55 | new | None | 0.0 | None
Summary: {'total_cases': 2, 'total_negative_delta': 250.0, 'total_positive_delta': -500.0, 'total_reserved': 22050.0, 'new': 2}
ERROR: no analyst
SUCCESS: Analyst assigned to PR-100
SUCCESS: Case PR-100 moved to investigating
SUCCESS: Reserve applied to PR-100
ERROR: abs(self.cases[case_id].delta_amount) > 0.01
ERROR: doesn`t investigate
ERROR: abs(self.cases[case_id].delta_amount) > 0.01
ERROR: not ready to close
SUCCESS: Case PR-102 created
SUCCESS: Analyst assigned to PR-102
SUCCESS: Case PR-102 moved to investigating
SUCCESS: Case PR-102 escalated
Partner payout cases:
PR-100 | best-gadgets | 2026-03 | 12500.0 | 11700.0 | 310 | -800.0 | 6.4 | reserve_applied | Olga | 300.0 | None
PR-101 | home-decor-plus | 2026-03 | 9800.0 | 10050.0 | 205 | 250.0 | 2.55 | new | No

In [ ]:
# Partner payout cases:
# PR-100 | best-gadgets | 2026-03 | 12500.0 | 12000.0 | 310 | -500.0 | 4.0 | new | None | 0.0 | None
# PR-101 | home-decor-plus | 2026-03 | 9800.0 | 10050.0 | 205 | 250.0 | 2.55 | new | None | 0.0 | None
# Summary: {'total_cases': 2, 'total_negative_delta': 500.0, 'total_positive_delta': 250.0, 'total_reserved': 0.0, 'new': 2}
# ERROR: Analyst is required
# SUCCESS: Analyst assigned to PR-100
# SUCCESS: Case PR-100 moved to investigating
# SUCCESS: Reserve applied to PR-100
# ERROR: Delta is not balanced
# ERROR: Case is not investigating
# ERROR: Delta is not balanced
# ERROR: Case is not ready to close
# SUCCESS: Case PR-102 created
# SUCCESS: Analyst assigned to PR-102
# SUCCESS: Case PR-102 moved to investigating
# SUCCESS: Case PR-102 escalated
# Partner payout cases:
# PR-100 | best-gadgets | 2026-03 | 12500.0 | 12300.0 | 310 | -200.0 | 1.6 | reserve_applied | Olga | 300.0 | None
# PR-101 | home-decor-plus | 2026-03 | 9800.0 | 10050.0 | 205 | 250.0 | 2.55 | new | None | 0.0 | None
# PR-102 | sport-zone | 2026-03 | 4300.0 | 3900.0 | 90 | -400.0 | 9.3 | escalated | Max | 0.0 | escalated
# Summary: {'total_cases': 3, 'total_negative_delta': 600.0, 'total_positive_delta': 250.0, 'total_reserved': 300.0, 'reserve_applied': 1, 'new': 1, 'escalated': 1}
#
# Финальное состояние:
# Partner payout cases:
# PR-100 | best-gadgets | 2026-03 | 12500.0 | 12300.0 | 310 | -200.0 | 1.6 | reserve_applied | Olga | 300.0 | None
# PR-101 | home-decor-plus | 2026-03 | 9800.0 | 10050.0 | 205 | 250.0 | 2.55 | new | None | 0.0 | None
# PR-102 | sport-zone | 2026-03 | 4300.0 | 3900.0 | 90 | -400.0 | 9.3 | escalated | Max | 0.0 | escalated
# Summary: {'total_cases': 3, 'total_negative_delta': 600.0, 'total_positive_delta': 250.0, 'total_reserved': 300.0, 'reserve_applied': 1, 'new': 1, 'escalated': 1}